In [1]:
#conda install geopandas

In [2]:
import pandas as pd
import quality_func

import geopandas as gpd

In [3]:
zone = quality_func.load_zone_data()
zone.shape

Set with labels : (265, 3)


(265, 3)

In [4]:
zone.columns = zone.columns.str.lower()

In [5]:
zone.head(5)

,locationid,borough,zone
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


In [6]:
files = ["data/uber-trip-data/uber-raw-data-apr14.csv", "data/uber-trip-data/uber-raw-data-may14.csv", "data/uber-trip-data/uber-raw-data-jun14.csv", 'data/uber-trip-data/uber-raw-data-jul14.csv', 'data/uber-trip-data/uber-raw-data-aug14.csv', 'data/uber-trip-data/uber-raw-data-sep14.csv']

raw_2014 = quality_func.load_2014_data(files)
raw_2014.shape

Set with labels : (4534327, 4)


(4534327, 4)

In [7]:
weekday_names = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
weekday_labels = dict(enumerate(weekday_names))

In [8]:
raw_2014.columns = raw_2014.columns.str.lower()
raw_2014["date/time"] = pd.to_datetime(raw_2014["date/time"], format="%m/%d/%Y %H:%M:%S")

In [9]:
raw_2014["weekday_number"] = raw_2014["date/time"].dt.dayofweek
raw_2014["weekday"] = raw_2014["weekday_number"].map(weekday_labels)
raw_2014["hour"] = raw_2014["date/time"].dt.hour

daily_counts_2014 = raw_2014.groupby(["weekday_number", "weekday"]).agg(
    pickups=("date/time", "size"),
    days=("date/time", "nunique"),
).reset_index().sort_values("weekday_number")

daily_counts_2014["pickups_per_day"] = daily_counts_2014["pickups"] / daily_counts_2014["days"]

raw_2014 = raw_2014.drop(columns=["weekday_number", "hour"])
raw_2014.columns = raw_2014.columns.str.lower()

In [10]:
raw_2014.head(5)

,date/time,lat,lon,base,weekday
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,Mardi
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,Mardi
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,Mardi
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,Mardi
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,Mardi


In [11]:
raw_2015 = quality_func.load_2015_data('data/uber-trip-data/uber-raw-data-janjune-15.csv')

Set with labels : (14270479, 4)


In [12]:
raw_2015.head(5)

,Dispatching_base_num,Pickup_date,Affiliated_base_num,locationID
0,B02617,2015-05-17 09:47:00,B02617,141
1,B02617,2015-05-17 09:47:00,B02617,65
2,B02617,2015-05-17 09:47:00,B02617,100
3,B02617,2015-05-17 09:47:00,B02774,80
4,B02617,2015-05-17 09:47:00,B02617,90


In [13]:
raw_2015 = raw_2015.rename(columns={"Pickup_date": "date/time", "Dispatching_base_num": "base"})
raw_2015["date/time"] = pd.to_datetime(raw_2015["date/time"], format="ISO8601")
raw_2015 = raw_2015.drop(columns=["Affiliated_base_num"])

raw_2015["weekday_number"] = raw_2015["date/time"].dt.dayofweek
raw_2015["weekday"] = raw_2015["weekday_number"].map(weekday_labels)
raw_2015["hour"] = raw_2015["date/time"].dt.hour

daily_counts_2015 = raw_2015.groupby(["weekday_number", "weekday"]).agg(
    pickups=("date/time", "size"),
    days=("date/time", "nunique"),
).reset_index().sort_values("weekday_number")

daily_counts_2015["pickups_per_day"] = daily_counts_2015["pickups"] / daily_counts_2015["days"]

raw_2015 = raw_2015.drop(columns=["weekday_number", "hour"])
raw_2015.columns = raw_2015.columns.str.lower()

# Explore dataset

In [14]:
dataframes_viz = [zone, raw_2014, raw_2015]
[print(f"Display of dataset: {display(df.head())}") for df in dataframes_viz]

,locationid,borough,zone
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


Display of dataset: None


,date/time,lat,lon,base,weekday
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,Mardi
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,Mardi
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,Mardi
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,Mardi
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,Mardi


Display of dataset: None


,base,date/time,locationid,weekday
0,B02617,2015-05-17 09:47:00,141,Dimanche
1,B02617,2015-05-17 09:47:00,65,Dimanche
2,B02617,2015-05-17 09:47:00,100,Dimanche
3,B02617,2015-05-17 09:47:00,80,Dimanche
4,B02617,2015-05-17 09:47:00,90,Dimanche


Display of dataset: None


[None, None, None]

In [15]:
zones_official = gpd.read_file("data/taxi_zones/taxi_zones/taxi_zones.shp")[["LocationID", "geometry"]].to_crs("EPSG:4326")

In [16]:
loc_2014 = gpd.GeoDataFrame(
    raw_2014,
    geometry=gpd.points_from_xy(raw_2014["lon"], raw_2014["lat"]),
    crs="EPSG:4326"
)

In [17]:
zones_official.columns = zones_official.columns.str.lower()

In [18]:
dataset_2014 = gpd.sjoin(loc_2014, zones_official, how="left", predicate="within").drop(columns=["index_right"])

In [19]:
dataset_2014.columns = dataset_2014.columns.str.lower()

In [20]:
dataset_2014.head(5)

,date/time,lat,lon,base,weekday,geometry,locationid
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,Mardi,POINT (-73.9549 40.769),140.0
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,Mardi,POINT (-74.0345 40.7267),NaN
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,Mardi,POINT (-73.9873 40.7316),79.0
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,Mardi,POINT (-73.9776 40.7588),161.0
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,Mardi,POINT (-73.9722 40.7594),162.0


In [21]:
dataset_2014.isnull().sum()

date/time         0
lat               0
lon               0
base              0
weekday           0
geometry          0
locationid    91085
dtype: int64

In [22]:
zones_official["rep_pt"] = zones_official.geometry.representative_point()
zones_official["lon"] = zones_official["rep_pt"].x
zones_official["lat"] = zones_official["rep_pt"].y

In [23]:
dataset_2015 = raw_2015.merge(zones_official[["locationid", "lat", "lon"]],on="locationid",how="left")

In [24]:
display(zone.head(1),zones_official.head(1) ,dataset_2014.head(1), dataset_2015.head(1))

,locationid,borough,zone
0,1,EWR,Newark Airport


,locationid,geometry,rep_pt,lon,lat
0,1,"POLYGON ((-74.18445 40.695, -74.18449 40.69509...",POINT (-74.17678 40.68951),-74.176778,40.689515


,date/time,lat,lon,base,weekday,geometry,locationid
0,2014-04-01 00:11:00,40.769,-73.9549,B02512,Mardi,POINT (-73.9549 40.769),140.0


,base,date/time,locationid,weekday,lat,lon
0,B02617,2015-05-17 09:47:00,141,Dimanche,40.766838,-73.959712


In [25]:
dataset_zone = pd.merge(zone, zones_official[["locationid", "lat", "lon"]], on="locationid", how="left")

In [26]:
dataset_2014 = dataset_2014.drop(columns=["geometry"])

In [27]:
display(dataset_zone.head(1), dataset_2014.head(1), dataset_2015.head(1))

,locationid,borough,zone,lat,lon
0,1,EWR,Newark Airport,40.689515,-74.176778


,date/time,lat,lon,base,weekday,locationid
0,2014-04-01 00:11:00,40.769,-73.9549,B02512,Mardi,140.0


,base,date/time,locationid,weekday,lat,lon
0,B02617,2015-05-17 09:47:00,141,Dimanche,40.766838,-73.959712


In [28]:
dataset_2014 = dataset_2014.rename(columns={"Date/Time": "date", "Lat": "lat", "Lon": "lon"})

In [29]:
dataset_2014.shape

(4534327, 6)

In [30]:
dataset_zone.value_counts("locationid")

locationid
1      1
183    1
169    1
170    1
171    1
      ..
95     1
96     1
97     1
98     1
265    1
Name: count, Length: 265, dtype: int64

In [31]:
dataset_2015.value_counts("locationid").head(10)

locationid
161    460732
231    420356
234    419045
79     407591
249    323989
230    315919
68     299781
163    296734
170    290550
132    289169
Name: count, dtype: int64

In [32]:
dataset_2014["locationid"] = dataset_2014["locationid"].astype("Int64")

In [33]:
dataset_2014.value_counts("locationid").head(10)

locationid
161    176704
234    146799
231    140461
79     125671
249    110826
68     109013
163    106837
162    105470
132    105083
230    105011
Name: count, dtype: int64

In [34]:
dataset = pd.concat([dataset_2014, dataset_2015], ignore_index=True)

In [35]:
dataset

,date/time,lat,lon,base,weekday,locationid
0,2014-04-01 00:11:00,40.769000,-73.954900,B02512,Mardi,140
1,2014-04-01 00:17:00,40.726700,-74.034500,B02512,Mardi,<NA>
2,2014-04-01 00:21:00,40.731600,-73.987300,B02512,Mardi,79
3,2014-04-01 00:28:00,40.758800,-73.977600,B02512,Mardi,161
4,2014-04-01 00:33:00,40.759400,-73.972200,B02512,Mardi,162
...,...,...,...,...,...,...
18804801,2015-05-08 15:43:00,40.748476,-73.992455,B02765,Vendredi,186
18804802,2015-05-08 15:43:00,40.778494,-73.951209,B02765,Vendredi,263
18804803,2015-05-08 15:43:00,40.742546,-73.996777,B02765,Vendredi,90
18804804,2015-05-08 15:44:00,40.713057,-73.998252,B02765,Vendredi,45


In [36]:
dataset = pd.merge(dataset, dataset_zone, how="left", on=["locationid"])

In [37]:
dataset = dataset.drop(columns=["lat_y", "lon_y"])

In [38]:
print(dataset.shape)
print(dataset.isnull().sum())
count = dataset.isnull().sum().sum() / len(dataset) * 100
print(f"Total number of missing values: {dataset.isnull().sum().sum()} sur {len(dataset)}")
print(f"Percentage of missing values: {count:.2f}%")

(18804806, 8)
date/time         0
lat_x          6264
lon_x          6264
base              0
weekday           0
locationid    91085
borough       91085
zone          91085
dtype: int64
Total number of missing values: 285783 sur 18804806
Percentage of missing values: 1.52%


In [39]:
dataset = dataset.dropna(subset=["lat_x", "lon_x", "borough", "zone", "locationid"])

In [40]:
print(dataset.isnull().sum())

date/time     0
lat_x         0
lon_x         0
base          0
weekday       0
locationid    0
borough       0
zone          0
dtype: int64


In [42]:
dataset.to_csv("data/dataset.csv", index=False)